# 01. Importing Libraries

In [208]:
import os
import math
import random
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import scipy
import numpy as np
import pandas as pd
import scipy.stats as st
from collections import Counter
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.preprocessing import MultiLabelBinarizer

# 02. Importing Data

In [115]:
books = pd.read_csv('../01. Data/books_combined.csv')

In [118]:
books.head()

,rank,title,author,avg_rating,num_ratings,cover_img_url,goodreads_url,source,year_published,genres,description,pages,series,isbn,publisher,language,ol_key,google_url
0,1.0,"The Hunger Games (The Hunger Games, #1)",Suzanne Collins,4.35,10115839.0,https://i.gr-assets.com/images/S/compressed.ph...,https://www.goodreads.com/book/show/2767052-th...,GoodReads,"First published September 14, 2008","Young Adult, Dystopia, Fiction, Fantasy, Scien...",Winning means fame and fortune. Losing means c...,"374 pages, Hardcover",The Hunger Games#1,NaN,NaN,NaN,NaN,NaN
1,2.0,Pride and Prejudice,Jane Austen,4.30,4920467.0,https://i.gr-assets.com/images/S/compressed.ph...,https://www.goodreads.com/book/show/1885.Pride...,GoodReads,"First published January 28, 1813","Classics, Romance, Fiction, Historical Fiction...","Since its immediate success in 1813,Pride and ...","279 pages, Paperback",NaN,NaN,NaN,NaN,NaN,NaN
2,3.0,To Kill a Mockingbird,Harper Lee,4.26,7016855.0,https://i.gr-assets.com/images/S/compressed.ph...,https://www.goodreads.com/book/show/2657.To_Ki...,GoodReads,"First published July 11, 1960","Classics, Fiction, Historical Fiction, School,...","""Shoot all the bluejays you want, if you can h...","323 pages, Paperback",To Kill a Mockingbird#1,NaN,NaN,NaN,NaN,NaN
3,4.0,Harry Potter and the Order of the Phoenix (Har...,J.K. Rowling,4.50,3868435.0,https://i.gr-assets.com/images/S/compressed.ph...,https://www.goodreads.com/book/show/58613451-h...,GoodReads,"First published June 21, 2003","Fantasy, Fiction, Young Adult, Harry Potter, M...",It's official: the evil Lord Voldemort has ret...,"896 pages, Hardcover",Harry Potter#5,NaN,NaN,NaN,NaN,NaN
4,5.0,The Book Thief,Markus Zusak,4.39,2935583.0,https://i.gr-assets.com/images/S/compressed.ph...,https://www.goodreads.com/book/show/19063.The_...,GoodReads,"First published September 1, 2005","Historical Fiction, Fiction, Young Adult, Clas...",Librarian's note: An alternate cover edition c...,"592 pages, Kindle Edition",NaN,NaN,NaN,NaN,NaN,NaN


In [119]:
books.shape

(2004, 18)

# 03. Cleaning the full dataframe 

In [120]:
books['rank'].describe()

count    1004.000000
mean      508.044821
std       292.110079
min         1.000000
25%       255.750000
50%       508.500000
75%       760.250000
max      1013.000000
Name: rank, dtype: float64

In [121]:
books['rank'].isnull().sum()

np.int64(1000)

In [122]:
# making a book_id column and dropping rank
books = books.reset_index(drop=True)
books.index.name = 'book_id'

books = books.drop(columns=['rank'])

In [123]:
books.head()

,title,author,avg_rating,num_ratings,cover_img_url,goodreads_url,source,year_published,genres,description,pages,series,isbn,publisher,language,ol_key,google_url
book_id,,,,,,,,,,,,,,,,,
0,"The Hunger Games (The Hunger Games, #1)",Suzanne Collins,4.35,10115839.0,https://i.gr-assets.com/images/S/compressed.ph...,https://www.goodreads.com/book/show/2767052-th...,GoodReads,"First published September 14, 2008","Young Adult, Dystopia, Fiction, Fantasy, Scien...",Winning means fame and fortune. Losing means c...,"374 pages, Hardcover",The Hunger Games#1,NaN,NaN,NaN,NaN,NaN
1,Pride and Prejudice,Jane Austen,4.30,4920467.0,https://i.gr-assets.com/images/S/compressed.ph...,https://www.goodreads.com/book/show/1885.Pride...,GoodReads,"First published January 28, 1813","Classics, Romance, Fiction, Historical Fiction...","Since its immediate success in 1813,Pride and ...","279 pages, Paperback",NaN,NaN,NaN,NaN,NaN,NaN
2,To Kill a Mockingbird,Harper Lee,4.26,7016855.0,https://i.gr-assets.com/images/S/compressed.ph...,https://www.goodreads.com/book/show/2657.To_Ki...,GoodReads,"First published July 11, 1960","Classics, Fiction, Historical Fiction, School,...","""Shoot all the bluejays you want, if you can h...","323 pages, Paperback",To Kill a Mockingbird#1,NaN,NaN,NaN,NaN,NaN
3,Harry Potter and the Order of the Phoenix (Har...,J.K. Rowling,4.50,3868435.0,https://i.gr-assets.com/images/S/compressed.ph...,https://www.goodreads.com/book/show/58613451-h...,GoodReads,"First published June 21, 2003","Fantasy, Fiction, Young Adult, Harry Potter, M...",It's official: the evil Lord Voldemort has ret...,"896 pages, Hardcover",Harry Potter#5,NaN,NaN,NaN,NaN,NaN
4,The Book Thief,Markus Zusak,4.39,2935583.0,https://i.gr-assets.com/images/S/compressed.ph...,https://www.goodreads.com/book/show/19063.The_...,GoodReads,"First published September 1, 2005","Historical Fiction, Fiction, Young Adult, Clas...",Librarian's note: An alternate cover edition c...,"592 pages, Kindle Edition",NaN,NaN,NaN,NaN,NaN,NaN


In [124]:
books.dtypes

title                 str
author                str
avg_rating        float64
num_ratings       float64
cover_img_url         str
goodreads_url         str
source                str
year_published        str
genres                str
description           str
pages                 str
series                str
isbn                  str
publisher             str
language              str
ol_key                str
google_url            str
dtype: object

In [125]:
# converting num_rating into int
books['num_ratings'] = books['num_ratings'].astype('Int64')

In [126]:
books.dtypes

title                 str
author                str
avg_rating        float64
num_ratings         Int64
cover_img_url         str
goodreads_url         str
source                str
year_published        str
genres                str
description           str
pages                 str
series                str
isbn                  str
publisher             str
language              str
ol_key                str
google_url            str
dtype: object

In [127]:
# checking for patterns in year_published column
books['year_published'].unique()

<ArrowStringArray>
['First published September 14, 2008',   'First published January 28, 1813',
      'First published July 11, 1960',      'First published June 21, 2003',
  'First published September 1, 2005',    'First published October 5, 2005',
    'First published August 17, 1945',    'First published January 1, 1954',
    'First published January 1, 1956',   'First published January 10, 2012',
 ...
                             '1200.0',                             '1788.0',
                             '1817.0',                             '1916.0',
                             '1874.0',                             '1835.0',
                             '1933.0',                             '1896.0',
                             '1934.0',                             '1829.0']
Length: 1054, dtype: str

In [128]:
# cleaning the year_published column
books['year_published'] = books['year_published'].str.extract(r'(\d{4})')

In [129]:
books['year_published'].unique()

<ArrowStringArray>
['2008', '1813', '1960', '2003', '2005', '1945', '1954', '1956', '2012',
 '1890',
 ...
 '1743', '1786', '1867', '1467', '1200', '1788', '1835', '1933', '1896',
 '1829']
Length: 245, dtype: str

In [130]:
books['year_published'].dtype

<StringDtype(na_value=nan)>

In [131]:
# changing it to number and checking it
books['year_published'] = pd.to_numeric(books['year_published'], errors='coerce').astype('Int64')
books['year_published'].dtype

Int64Dtype()

In [132]:
# checking for patterns in pages column
books['pages'].unique()

<ArrowStringArray>
[             '374 pages, Hardcover',              '279 pages, Paperback',
              '323 pages, Paperback',              '896 pages, Hardcover',
         '592 pages, Kindle Edition',              '498 pages, Paperback',
  '141 pages, Mass Market Paperback', '1728 pages, Mass Market Paperback',
              '767 pages, Paperback',              '313 pages, Hardcover',
 ...
                             '684.0',                             '736.0',
                             '770.0',                             '380.0',
                             '590.0',                             '884.0',
                             '958.0',                             '874.0',
                             '749.0',                            '1856.0']
Length: 1189, dtype: str

In [133]:
# cleaning the pages column
books['pages'] = books['pages'].str.extract(r'(\d+)')

In [134]:
books['pages'].unique()

<ArrowStringArray>
[ '374',  '279',  '323',  '896',  '592',  '498',  '141', '1728',  '767',
  '313',
 ...
  '177',  '902', '1232',  '575',  '770',  '884',  '958',  '874',  '749',
 '1856']
Length: 669, dtype: str

In [135]:
books['pages'].dtype

<StringDtype(na_value=nan)>

In [136]:
# changing it to number and checking it
books['pages'] = pd.to_numeric(books['pages'], errors='coerce').astype('Int64')
books['pages'].dtype

Int64Dtype()

Manual imputation for some important data

In [138]:
# checking columns that will need manual imputation 
books.isnull().sum()

title                0
author               1
avg_rating         755
num_ratings        733
cover_img_url        4
goodreads_url     1000
source               0
year_published      20
genres               7
description          0
pages               24
series            1529
isbn              1008
publisher         1009
language          1004
ol_key            1004
google_url        1017
dtype: int64

In [139]:
# checking the rows with null to be able to search and manually fill them
books[books['year_published'].isnull()]

,title,author,avg_rating,num_ratings,cover_img_url,goodreads_url,source,year_published,genres,description,pages,series,isbn,publisher,language,ol_key,google_url
book_id,,,,,,,,,,,,,,,,,
48,Harry Potter and the Deathly Hallows (Harry Po...,J.K. Rowling,4.62,4177757,https://i.gr-assets.com/images/S/compressed.ph...,https://www.goodreads.com/book/show/136251.Har...,GoodReads,<NA>,NaN,page unavailableAn unexpected error occurred. ...,<NA>,NaN,NaN,NaN,NaN,NaN,NaN
50,The Odyssey,Homer,3.84,1224393,https://i.gr-assets.com/images/S/compressed.ph...,https://www.goodreads.com/book/show/1381.The_O...,GoodReads,<NA>,"Classics, Fiction, Poetry, Mythology, Fantasy,...","Sing to me of the man, Muse, the man of twists...",541,NaN,NaN,NaN,NaN,NaN,NaN
72,The Adventures of Sherlock Holmes (Sherlock Ho...,Arthur Conan Doyle,4.30,330762,https://i.gr-assets.com/images/S/compressed.ph...,https://www.goodreads.com/book/show/3590.The_A...,GoodReads,<NA>,NaN,page unavailableAn unexpected error occurred. ...,<NA>,NaN,NaN,NaN,NaN,NaN,NaN
90,Gone Girl,Gillian Flynn,4.15,3508073,https://i.gr-assets.com/images/S/compressed.ph...,https://www.goodreads.com/book/show/19288043-g...,GoodReads,<NA>,NaN,page unavailableAn unexpected error occurred. ...,<NA>,NaN,NaN,NaN,NaN,NaN,NaN
152,"The Last Thing He Told Me (Hannah Hall, #1)",Laura Dave,3.83,1187142,https://i.gr-assets.com/images/S/compressed.ph...,https://www.goodreads.com/book/show/54981009-t...,GoodReads,<NA>,NaN,page unavailableAn unexpected error occurred. ...,<NA>,NaN,NaN,NaN,NaN,NaN,NaN
275,Beloved,Toni Morrison,3.99,503338,https://i.gr-assets.com/images/S/compressed.ph...,https://www.goodreads.com/book/show/6149.Beloved,GoodReads,<NA>,NaN,page unavailableAn unexpected error occurred. ...,<NA>,NaN,NaN,NaN,NaN,NaN,NaN
291,The Iliad / The Odyssey,Homer,4.07,85200,https://i.gr-assets.com/images/S/compressed.ph...,https://www.goodreads.com/book/show/1375.The_I...,GoodReads,<NA>,"Classics, Fiction, Poetry, Mythology, Fantasy,...",Gripping listeners and readers for more than 2...,1556,NaN,NaN,NaN,NaN,NaN,NaN
470,Tao Te Ching,Lao Tzu,4.29,185834,https://i.gr-assets.com/images/S/compressed.ph...,https://www.goodreads.com/book/show/67896.Tao_...,GoodReads,<NA>,"Philosophy, Nonfiction, Classics, Spirituality...",A lucid translation of the well-known Taoist c...,107,SUNY series in Chinese Philosophy and Culture,NaN,NaN,NaN,NaN,NaN
476,القرآن الكريم,الله,4.38,73505,https://i.gr-assets.com/images/S/compressed.ph...,https://www.goodreads.com/book/show/646462._,GoodReads,<NA>,"Religion, Islam, Nonfiction, Classics, Philoso...",The Quran (English pronunciation: /kɔrˈɑːn/; A...,604,NaN,NaN,NaN,NaN,NaN,NaN


In [140]:
# changing year_published to object to be able to fill the null values with the BC values
books['year_published'] = books['year_published'].astype('object')

In [ ]:
# substituteing the null values with the year of the first edition of the book
# AC
books.loc[48, 'year_published'] = 2007 
books.loc[72, 'year_published'] = 1892 
books.loc[90, 'year_published'] = 2012   
books.loc[152, 'year_published'] = 2021  
books.loc[275, 'year_published'] = 1987  
books.loc[518, 'year_published'] = 2022  
books.loc[563, 'year_published'] = 2003  
books.loc[476, 'year_published'] = 632   
books.loc[866, 'year_published'] = 180   
books.loc[1710, 'year_published'] = 1897

# BC
books.loc[50, 'year_published'] = '800 BC'    
books.loc[291, 'year_published'] = '800 BC'   
books.loc[470, 'year_published'] = '400 BC'   
books.loc[585, 'year_published'] = '375 BC'   
books.loc[685, 'year_published'] = '500 BC'   
books.loc[727, 'year_published'] = '800 BC'  
books.loc[768, 'year_published'] = '800 BC'   
books.loc[795, 'year_published'] = '429 BC'   
books.loc[880, 'year_published'] = '19 BC'    
books.loc[928, 'year_published'] = '441 BC'   

In [142]:
# checking rows with null values in genres column
books[books['genres'].isnull()]

,title,author,avg_rating,num_ratings,cover_img_url,goodreads_url,source,year_published,genres,description,pages,series,isbn,publisher,language,ol_key,google_url
book_id,,,,,,,,,,,,,,,,,
48,Harry Potter and the Deathly Hallows (Harry Po...,J.K. Rowling,4.62,4177757,https://i.gr-assets.com/images/S/compressed.ph...,https://www.goodreads.com/book/show/136251.Har...,GoodReads,2007,NaN,page unavailableAn unexpected error occurred. ...,<NA>,NaN,NaN,NaN,NaN,NaN,NaN
72,The Adventures of Sherlock Holmes (Sherlock Ho...,Arthur Conan Doyle,4.30,330762,https://i.gr-assets.com/images/S/compressed.ph...,https://www.goodreads.com/book/show/3590.The_A...,GoodReads,1892,NaN,page unavailableAn unexpected error occurred. ...,<NA>,NaN,NaN,NaN,NaN,NaN,NaN
90,Gone Girl,Gillian Flynn,4.15,3508073,https://i.gr-assets.com/images/S/compressed.ph...,https://www.goodreads.com/book/show/19288043-g...,GoodReads,2012,NaN,page unavailableAn unexpected error occurred. ...,<NA>,NaN,NaN,NaN,NaN,NaN,NaN
152,"The Last Thing He Told Me (Hannah Hall, #1)",Laura Dave,3.83,1187142,https://i.gr-assets.com/images/S/compressed.ph...,https://www.goodreads.com/book/show/54981009-t...,GoodReads,2021,NaN,page unavailableAn unexpected error occurred. ...,<NA>,NaN,NaN,NaN,NaN,NaN,NaN
275,Beloved,Toni Morrison,3.99,503338,https://i.gr-assets.com/images/S/compressed.ph...,https://www.goodreads.com/book/show/6149.Beloved,GoodReads,1987,NaN,page unavailableAn unexpected error occurred. ...,<NA>,NaN,NaN,NaN,NaN,NaN,NaN
518,Hidden Pictures,Jason Rekulak,4.14,646390,https://i.gr-assets.com/images/S/compressed.ph...,https://www.goodreads.com/book/show/58724923-h...,GoodReads,2022,NaN,page unavailableAn unexpected error occurred. ...,<NA>,NaN,NaN,NaN,NaN,NaN,NaN
563,Shantaram,Gregory David Roberts,4.28,244510,https://i.gr-assets.com/images/S/compressed.ph...,https://www.goodreads.com/book/show/33600.Shan...,GoodReads,2003,NaN,page unavailableAn unexpected error occurred. ...,<NA>,NaN,NaN,NaN,NaN,NaN,NaN


In [143]:
pd.set_option('display.max_colwidth', None)

In [146]:
# checking some collections genres to generate the same input for the null values of same collection
# HP books
books[books['title'].str.contains('Harry Potter', case=False)][['title', 'genres']]

,title,genres
book_id,,
3,"Harry Potter and the Order of the Phoenix (Harry Potter, #5)","Fantasy, Fiction, Young Adult, Harry Potter, Magic, Audiobook, Childrens"
48,"Harry Potter and the Deathly Hallows (Harry Potter, #7)",NaN
56,"Harry Potter and the Prisoner of Azkaban (Harry Potter, #3)","Fantasy, Fiction, Young Adult, Harry Potter, Magic, Audiobook, Childrens"
69,"Harry Potter and the Philosopher's Stone (Harry Potter, #1)","Fantasy, Fiction, Young Adult, Harry Potter, Magic, Audiobook, Childrens"
89,"Harry Potter and the Goblet of Fire (Harry Potter, #4)","Fantasy, Fiction, Young Adult, Harry Potter, Magic, Audiobook, Childrens"
104,"Harry Potter and the Half-Blood Prince (Harry Potter, #6)","Fantasy, Fiction, Young Adult, Harry Potter, Magic, Audiobook, Childrens"
109,"Harry Potter and the Sorcerer's Stone (Harry Potter, #1)","Fantasy, Fiction, Young Adult, Harry Potter, Magic, Audiobook, Childrens"
117,"Harry Potter and the Chamber of Secrets (Harry Potter, #2)","Fantasy, Fiction, Young Adult, Harry Potter, Magic, Audiobook, Childrens"
439,"Harry Potter Series Box Set (Harry Potter, #1-7)","Fantasy, Young Adult, Fiction, Harry Potter, Magic, Childrens, Adventure"


In [147]:
# Sherlock Holmes books
books[books['title'].str.contains('Sherlock Holmes', case=False)][['title', 'genres']]

,title,genres
book_id,,
72,"The Adventures of Sherlock Holmes (Sherlock Holmes, #3)",NaN
350,The Complete Sherlock Holmes,"Classics, Mystery, Fiction, Crime, Short Stories, Audiobook, Detective"
413,"The Hound of the Baskervilles (Sherlock Holmes, #5)","Classics, Mystery, Fiction, Crime, Detective, Mystery Thriller, Thriller"
1835,The Devil and Sherlock Holmes,"Literary Criticism, Nonfiction, True crime stories, Case studies, murder"


In [148]:
# Hannah Hall books
books[books['title'].str.contains('Hannah Hall', case=False)][['title', 'genres']]

,title,genres
book_id,,
152,"The Last Thing He Told Me (Hannah Hall, #1)",NaN


In [149]:
# imputting genres manually based on the title and author of the book
books.loc[48, 'genres'] = 'Fantasy, Fiction, Young Adult, Harry Potter, Magic, Audiobook, Childrens'
books.loc[72, 'genres'] = 'Classics, Mystery, Fiction, Crime, Short Stories, Audiobook, Detective'
books.loc[90, 'genres'] = 'Mystery, Thriller, Fiction, Suspense'
books.loc[152, 'genres'] = 'Mystery, Thriller, Fiction, Suspense'
books.loc[275, 'genres'] = 'Fiction, Historical Fiction, Classics, Literary'
books.loc[518, 'genres'] = 'Horror, Mystery, Fiction, Thriller'
books.loc[563, 'genres'] = 'Biography, Adventure, Historical Fiction, Travel'

In [150]:
# checking how many rows have description "page unavailable"
books['description'].str.contains('page unavailable').sum()

np.int64(7)

In [151]:
# checking which rows for manual imputation of description
books[books['description'].str.contains('page unavailable', case=False)][['title', 'author', 'description', 'source']]

,title,author,description,source
book_id,,,,
48,"Harry Potter and the Deathly Hallows (Harry Potter, #7)",J.K. Rowling,page unavailableAn unexpected error occurred. We will investigate this problem as soon as possible — please check back soon!,GoodReads
72,"The Adventures of Sherlock Holmes (Sherlock Holmes, #3)",Arthur Conan Doyle,page unavailableAn unexpected error occurred. We will investigate this problem as soon as possible — please check back soon!,GoodReads
90,Gone Girl,Gillian Flynn,page unavailableAn unexpected error occurred. We will investigate this problem as soon as possible — please check back soon!,GoodReads
152,"The Last Thing He Told Me (Hannah Hall, #1)",Laura Dave,page unavailableAn unexpected error occurred. We will investigate this problem as soon as possible — please check back soon!,GoodReads
275,Beloved,Toni Morrison,page unavailableAn unexpected error occurred. We will investigate this problem as soon as possible — please check back soon!,GoodReads
518,Hidden Pictures,Jason Rekulak,page unavailableAn unexpected error occurred. We will investigate this problem as soon as possible — please check back soon!,GoodReads
563,Shantaram,Gregory David Roberts,page unavailableAn unexpected error occurred. We will investigate this problem as soon as possible — please check back soon!,GoodReads


In [152]:
# manual imputations according to source
books.loc[48, 'description'] = "It's no longer safe for Harry at Hogwarts, so he and his best friends, Ron and Hermione, are on the run. Professor Dumbledore has given them clues about what they need to do to defeat the dark wizard, Lord Voldemort, once and for all, but it's up to them to figure out what these hints and suggestions really mean. Their cross-country odyssey has them searching desperately for the answers, while evading capture or death at every turn. At the same time, their friendship, fortitude, and sense of right and wrong are tested in ways they never could have imagined. The ultimate battle between good and evil that closes out this final chapter of the epic series takes place where Harry's Wizarding life began: at Hogwarts. The satisfying conclusion offers shocking last-minute twists, incredible acts of courage, powerful new forms of magic, and the resolution of many mysteries. Above all, this intense, cathartic book serves as a clear statement of the message at the heart of the Harry Potter series: that choice matters much more than destiny, and that love will always triumph over death."
books.loc[72, 'description'] = 'The world-famous crime-solving duo Holmes and Watson are on sterling form in this excellent compilation of six unabridged adventures. Included in the third volume of the audio collection read by famed actor Edward Hardwicke are: "A Scandal in Bohemia," which inspired the Guy Richie-directed Sherlock Holmes film starring Robert Downey Junior; "Silver Blaze," in which a racehorse disappears on the eve of an important meet and its trainer appears to have been murdered; "The Adventure of the Copper Beeches," in which a young woman agrees to a governess post with an impressive salary but strange conditions; as well as "The Adventure of the Priory School," "The Red-Headed League," and "The Adventure of the Blue Carbuncle."'
books.loc[90, 'description'] = "Who are you? What have we done to each other? These are the questions Nick Dunne finds himself asking on the morning of his fifth wedding anniversary when his wife Amy suddenly disappears. The police suspect Nick. Amy's friends reveal that she was afraid of him, that she kept secrets from him. He swears it isn't true. A police examination of his computer shows strange searches. He says they weren't made by him. And then there are the persistent calls on his mobile phone. So what did happen to Nick's beautiful wife?"
books.loc[152, 'description'] = "Before Owen Michaels disappears, he smuggles a note to his beloved wife of one year: Protect her. Despite her confusion and fear, Hannah Hall knows exactly to whom the note refers—Owen’s sixteen-year-old daughter, Bailey. Bailey, who lost her mother tragically as a child. Bailey, who wants absolutely nothing to do with her new stepmother. As Hannah’s increasingly desperate calls to Owen go unanswered, as the FBI arrests Owen’s boss, as a US marshal and federal agents arrive at her Sausalito home unannounced, Hannah quickly realizes her husband isn’t who he said he was. And that Bailey just may hold the key to figuring out Owen’s true identity—and why he really disappeared. Hannah and Bailey set out to discover the truth. But as they start putting together the pieces of Owen’s past, they soon realize they’re also building a new future—one neither of them could have anticipated."
books.loc[275, 'description'] = "Staring unflinchingly into the abyss of slavery, this spellbinding novel transforms history into a story as powerful as Exodus and as intimate as a lullaby. Sethe, its protagonist, was born a slave and escaped to Ohio, but eighteen years later she is still not free. She has too many memories of Sweet Home, the beautiful farm where so many hideous things happened. And Sethe's new home is haunted by the ghost of her baby, who died nameless and whose tombstone is engraved with a single word: Beloved. Filled with bitter poetry and suspense as taut as a rope, Beloved is a towering achievement by Nobel Prize laureate Toni Morrison."
books.loc[518, 'description'] = "From Jason Rekulak, Edgar-nominated author of The Impossible Fortress, comes a wildly inventive spin on the classic horror story in Hidden Pictures, a supernatural thriller about a woman working as a nanny for a young boy with strange and disturbing secrets. Fresh out of rehab, Mallory Quinn takes a job as a babysitter for Ted and Caroline Maxwell. She is to look after their five-year-old son, Teddy. Mallory immediately loves it. She has her own living space, goes out for nightly runs, and has the stability she craves. And she sincerely bonds with Teddy, a sweet, shy boy who is never without his sketchbook and pencil. His drawings are the usual fare: trees, rabbits, balloons. But one day, he draws something different: a man in a forest, dragging a woman’s lifeless body. Then, Teddy’s artwork becomes increasingly sinister, and his stick figures quickly evolve into lifelike sketches well beyond the ability of any five-year-old. Mallory begins to wonder if these are glimpses of a long-unsolved murder, perhaps relayed by a supernatural force. Knowing just how crazy it all sounds, Mallory nevertheless sets out to decipher the images and save Teddy before it’s too late."
books.loc[563, 'description'] = """It took me a long time and most of the world to learn what I know about love and fate and the choices we make, but the heart of it came to me in an instant, while I was chained to a wall and being tortured. So begins this epic, mesmerizing first novel set in the underworld of contemporary Bombay. Shantaram is narrated by Lin, an escaped convict with a false passport who flees maximum security prison in Australia for the teeming streets of a city where he can disappear. Accompanied by his guide and faithful friend, Prabaker, the two enter Bombay's hidden society of beggars and gangsters, prostitutes and holy men, soldiers and actors, and Indians and exiles from other countries, who seek in this remarkable place what they cannot find elsewhere. As a hunted man without a home, family, or identity, Lin searches for love and meaning while running a clinic in one of the city's poorest slums, and serving his apprenticeship in the dark arts of the Bombay mafia. Based on the life of the author, it is by any measure the debut of an extraordinary voice in literature."""

In [153]:
books[books['author'].isnull()]

,title,author,avg_rating,num_ratings,cover_img_url,goodreads_url,source,year_published,genres,description,pages,series,isbn,publisher,language,ol_key,google_url
book_id,,,,,,,,,,,,,,,,,
1240,The business world,NaN,NaN,<NA>,NaN,NaN,Open Library,1912,"Great Britain -- Commerce -- History., Great Britain -- Industries -- History.","Accounts of W.V. Bowater and Sons; George Briggs and Co., alderman and sheriff; C.A. Hanson, C.C. Wakefield and Co.",336,NaN,9781501500725,Dod's Publications,eng,/works/OL19729600M,https://play.google.com/store/books/details?id=NvBeCAAAQBAJ&source=gbs_api


In [154]:
# manual inputation of author name (first one)
books.loc[1240, 'author'] = 'Dwi Noverini Djenar'

In [155]:
books.isnull().sum()

title                0
author               0
avg_rating         755
num_ratings        733
cover_img_url        4
goodreads_url     1000
source               0
year_published       0
genres               0
description          0
pages               24
series            1529
isbn              1008
publisher         1009
language          1004
ol_key            1004
google_url        1017
dtype: int64

In [156]:
pd.reset_option('display.max_colwidth')

In [ ]:
# exporting the cleaned dataframe as is
# books.to_csv('../01. Data/books_cleaned.csv', index=True)

# 04. Transformations for ML

In [212]:
# creating a copy of the dataframe to work with
books_model = books.copy()

In [185]:
# dropping columns that wont be used for the ML model (year not used due to object values for BC dates)
columns_to_drop_clustering = ['cover_img_url', 'goodreads_url', 'source', 'series', 'isbn', 'publisher', 'language', 'ol_key', 'google_url', 'year_published']

books_model = books_model.drop(columns=columns_to_drop_clustering)

In [186]:
books_model.head()

,title,author,avg_rating,num_ratings,genres,description,pages
book_id,,,,,,,
0,"The Hunger Games (The Hunger Games, #1)",Suzanne Collins,4.35,10115839,"Young Adult, Dystopia, Fiction, Fantasy, Scien...",Winning means fame and fortune. Losing means c...,374
1,Pride and Prejudice,Jane Austen,4.30,4920467,"Classics, Romance, Fiction, Historical Fiction...","Since its immediate success in 1813,Pride and ...",279
2,To Kill a Mockingbird,Harper Lee,4.26,7016855,"Classics, Fiction, Historical Fiction, School,...","""Shoot all the bluejays you want, if you can h...",323
3,Harry Potter and the Order of the Phoenix (Har...,J.K. Rowling,4.50,3868435,"Fantasy, Fiction, Young Adult, Harry Potter, M...",It's official: the evil Lord Voldemort has ret...,896
4,The Book Thief,Markus Zusak,4.39,2935583,"Historical Fiction, Fiction, Young Adult, Clas...",Librarian's note: An alternate cover edition c...,592


In [187]:
books_model['author'].nunique()

1377

In [188]:
# dropping title and author as those are identifiers. Author dropped because there are to many unique values and it would be hard to encode them
books_model = books_model.drop(columns=['title', 'author'])

In [189]:
books_model.head()

,avg_rating,num_ratings,genres,description,pages
book_id,,,,,
0,4.35,10115839,"Young Adult, Dystopia, Fiction, Fantasy, Scien...",Winning means fame and fortune. Losing means c...,374
1,4.30,4920467,"Classics, Romance, Fiction, Historical Fiction...","Since its immediate success in 1813,Pride and ...",279
2,4.26,7016855,"Classics, Fiction, Historical Fiction, School,...","""Shoot all the bluejays you want, if you can h...",323
3,4.50,3868435,"Fantasy, Fiction, Young Adult, Harry Potter, M...",It's official: the evil Lord Voldemort has ret...,896
4,4.39,2935583,"Historical Fiction, Fiction, Young Adult, Clas...",Librarian's note: An alternate cover edition c...,592


In [190]:
books_model.isnull().sum()

avg_rating     755
num_ratings    733
genres           0
description      0
pages           24
dtype: int64

In [191]:
# filling null in avg_rating and num_ratings with median of each column
books_model['avg_rating'] = books_model['avg_rating'].fillna(books_model['avg_rating'].median())
books_model['num_ratings'] = books_model['num_ratings'].fillna(books_model['num_ratings'].median())
books_model['pages'] = books_model['pages'].fillna(round(books_model['pages'].median()))

In [192]:
books_model.isnull().sum()

avg_rating     0
num_ratings    0
genres         0
description    0
pages          0
dtype: int64

Running TF-IDF and PCA in description column

In [193]:
# running the TF-IDF vectorizer on description column
tfidf = TfidfVectorizer(max_features=5000)
X_description = tfidf.fit_transform(books_model['description']).toarray()
print('TF-IDF shape:', X_description.shape)

TF-IDF shape: (2004, 5000)


In [194]:
# running PCA to reduce the dimensionality of the TF-IDF created matrix
pca = PCA(n_components=50)
X_description_pca = pca.fit_transform(X_description)
print('PCA shape:', X_description_pca.shape)

PCA shape: (2004, 50)


OHE on genre column

In [209]:
# creating new column with the list of genres for each book
def split_genres(genres_string):
    if pd.isna(genres_string):
        return []
    genres = re.split(r'[,;|]', genres_string)
    cleaned = []
    for genre in genres:
        cleaned.append(genre.strip())
    return cleaned

books_model['genres_list'] = books_model['genres'].apply(split_genres)

In [210]:
# cleaning and standardizing the genres column_list
def clean_genres(genre_list):
    cleaned = []
    for genre in genre_list:
        cleaned.append(genre.strip().title())
    return cleaned

books_model['genres_list'] = books_model['genres_list'].apply(clean_genres)

In [211]:
print(genre_counts.most_common())

[('Fiction', 1160), ('Classics', 443), ('Fantasy', 423), ('Young Adult', 302), ('Romance', 291), ('Audiobook', 253), ('Historical Fiction', 232), ('Book Club', 199), ('Literature', 196), ('Mystery', 196), ('Contemporary', 181), ('Thriller', 165), ('Nonfiction', 161), ('Science Fiction', 151), ('Novels', 136), ('History', 128), ('Adventure', 121), ('Biography', 119), ('Historical', 112), ('Philosophy', 110), ('Childrens', 102), ('Mystery Thriller', 97), ('Crime', 97), ('Horror', 89), ('Paranormal', 81), ('Adult', 71), ('Magic', 70), ('Dystopia', 69), ('Middle Grade', 67), ('Suspense', 65), ('Literary Fiction', 64), ('Drama', 63), ('School', 60), ('general', 58), ('fiction', 54), ('Humor', 52), ('Vampires', 49), ('Poetry', 49), ('Psychology', 49), ('Science Fiction Fantasy', 46), ('Urban Fantasy', 46), ('True Crime', 46), ('science fiction', 46), ('Murder', 41), ('Travel', 40), ('Science fiction', 40), ('Description and travel', 37), ('Religion', 36), ('Business', 35), ('High Fantasy', 3

In [177]:
# running the MultiLabelBinarizer on genres column
mlb = MultiLabelBinarizer()
X_genres = mlb.fit_transform(books_model['genres_list'])
print('Genres shape:', X_genres.shape)

Genres shape: (2004, 2425)


Normalizing avg_rating, num_ratings, pages